# Product Recommendation with FAISS
**Goal:** recommend products similar to the ones a user has already viewed.

Steps:
1. Turn each product description into an **embedding** with `all-MiniLM-L6-v2`.
2. Store the embeddings in a FAISS index that uses **L2 distance** (smaller distance = more similar).
3. For every product the user viewed, find its nearest neighbors, combine the results and remove products the user has already seen.

In [ ]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

## Our data
A small product catalog and the product IDs the user has viewed.

In [ ]:
df = pd.DataFrame({
    "product_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "name": [
        "Classic Blue Jeans", "Slim Fit Denim Jacket", "White Cotton T-Shirt",
        "Running Shoes", "Leather Ankle Boots", "Black Leather Handbag",
        "Wool Winter Scarf", "Knitted Wool Sweater", "Sports Training Shorts",
        "Canvas Sneakers",
    ],
    "description": [
        "Comfortable blue denim jeans with a straight cut",
        "Stylish denim jacket with a slim fit and metal buttons",
        "Soft white t-shirt made of breathable cotton",
        "Lightweight running shoes with cushioned soles for jogging",
        "Elegant ankle boots made of genuine leather",
        "Spacious black handbag made of premium leather",
        "Warm scarf made of soft wool for cold winter days",
        "Cozy knitted sweater made of thick wool",
        "Breathable shorts for gym workouts and sports training",
        "Casual canvas sneakers for everyday wear",
    ],
})

user_browsing_history = [101, 104]   # product IDs the user has viewed
df

## Step 1: Create embeddings and the FAISS index

In [ ]:
def create_embeddings(df):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    descriptions = df["description"].tolist()
    embeddings = model.encode(descriptions)
    embeddings = np.array(embeddings, dtype="float32")  # FAISS needs float32

    index = faiss.IndexFlatL2(embeddings.shape[1])      # L2 distance index
    index.add(embeddings)
    return model, index, embeddings

## Step 2: Find recommendations
For each viewed product we search for its `k` nearest products. The closest match is always the product itself, so we remove viewed products at the end.

In [ ]:
def find_rec_products(df, user_browsing_history, embeddings, index, k=3):
    # Row positions of the viewed products in the dataframe
    viewed_rows = df.index[df["product_id"].isin(user_browsing_history)].tolist()

    # Embeddings of the viewed products
    viewed_embeddings = embeddings[viewed_rows]

    # k nearest products for each viewed product
    distances, indices = index.search(viewed_embeddings, k)

    # Combine results and keep each product only once
    recommended_rows = set(indices.flatten())

    # Remove products the user has already viewed
    recommended_rows = recommended_rows - set(viewed_rows)

    # Look up product IDs and names
    result = df.loc[sorted(recommended_rows), ["product_id", "name"]]
    return result.reset_index(drop=True)

## Run it

In [ ]:
def main():
    model, index, embeddings = create_embeddings(df)
    recommendations = find_rec_products(df, user_browsing_history, embeddings, index, k=3)
    print("Recommended products:")
    print(recommendations)

main()

## Try it yourself
- Change `user_browsing_history` (for example `[107]`). Do the recommendations match the style of the product?
- Change `k` to 5 and see how the list grows.
- Add your own products to the table.